# vLLM + TensorRT on a Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbhinandanMishra41/multimodal-transformer-benchmark/blob/main/notebooks/Colab_Full_Benchmark_vLLM_TRT.ipynb)

vLLM only runs on Linux, so this notebook does that part on a Colab GPU (a T4 is fine). It also runs the full benchmark there, which picks up the TensorRT execution provider for the vision encoder when it's available.

vLLM only sees the text decoder (there's no vision tower in the exported checkpoint), so the vLLM part is a text-only throughput + correctness check against my own decoder.

In [ ]:
!nvidia-smi
!git clone https://github.com/AbhinandanMishra41/multimodal-transformer-benchmark.git
%cd multimodal-transformer-benchmark
!pip install -q vllm "onnx>=1.15" "onnxruntime-gpu>=1.17" "tensorrt>=10" matplotlib pytest

In [ ]:
import os, sys, glob
sys.path.insert(0, "src")
# the tensorrt wheel keeps libnvinfer in site-packages; onnxruntime's TensorRT EP needs it on the path
trt_libs = glob.glob("/usr/local/lib/python3*/dist-packages/tensorrt_libs")
if trt_libs:
    os.environ["LD_LIBRARY_PATH"] = trt_libs[0] + ":" + os.environ.get("LD_LIBRARY_PATH", "")

import torch
print(torch.__version__, torch.cuda.get_device_name(0))

## Tests

In [ ]:
!python -m pytest -q

## Full benchmark

Writes `results/` and `assets/`.

In [ ]:
from benchmark.runner import run_full_benchmark
report = run_full_benchmark(profile="compact", max_new_tokens=32, train_steps=300)
report["tensorrt_vision_encoder"]

## Building a TensorRT engine directly

Same vision encoder, through the TensorRT builder API instead of ONNX Runtime.

In [ ]:
from quantization.tensorrt_engine import build_vision_engine, detect_tensorrt_support
from vlm.config import VLMConfig

print(detect_tensorrt_support())
engine = build_vision_engine(
    "artifacts/compact/onnx/vision_projector_fp32.onnx",
    "artifacts/compact/trt/vision_projector_fp16.engine",
    VLMConfig.compact(),
)
print(engine, f"{engine.stat().st_size / 1e6:.1f} MB")

## vLLM

Loads `artifacts/compact/vllm_hf_checkpoint` as a `LlamaForCausalLM`, generates for 16 prompts, and compares the first few with my own decoder's greedy output.

In [ ]:
!PYTHONPATH=src python -m benchmark.vllm_run --checkpoint artifacts/compact/vllm_hf_checkpoint --batch-size 16